# 04 - Modele ML: Linear Regression si Random Forest

Scop: prezicem `arr_delay` folosind Spark MLlib. Comparam Linear Regression cu Random Forest Regression folosind RMSE, MAE si R2.

Codul este KISS si foloseste doar biblioteci disponibile in Dockerfile.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

spark = (
    SparkSession.builder
    .appName("FlightsProject")
    .master("spark://master:7077")
    .config("spark.executor.memory", "1g")
    .config("spark.executor.cores", "1")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator


## Incarcare date curatate


In [ ]:
CLEAN_CSV_PATH = "hdfs://master:9000/flights/processed/flights_clean_csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CLEAN_CSV_PATH)
    .repartition(4)
)

# Selectam doar coloane utile pentru model.
model_df = df.select(
    "arr_delay", "dep_delay", "air_time", "distance", "hour", "month", "carrier", "origin", "dest"
).dropna()

# Esantion optional pentru laptop slab. Daca merge repede, comentati linia sample.
model_df = model_df.sample(False, 0.35, seed=42)
model_df = model_df.cache()

print("Randuri pentru ML:", model_df.count())
model_df.show(5, truncate=False)


## Train-test split


In [ ]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
print("Train:", train_df.count())
print("Test:", test_df.count())


## Pipeline de feature engineering

Categoricele sunt transformate cu StringIndexer + OneHotEncoder. Numericele sunt puse direct in vectorul de features.


In [ ]:
categorical_cols = ["carrier", "origin", "dest"]
numeric_cols = ["dep_delay", "air_time", "distance", "hour", "month"]

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_cols
]
encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in categorical_cols
]

assembler = VectorAssembler(
    inputCols=numeric_cols + [f"{c}_ohe" for c in categorical_cols],
    outputCol="features",
    handleInvalid="keep"
)


## Model 1: Linear Regression


In [ ]:
lr = LinearRegression(
    featuresCol="features",
    labelCol="arr_delay",
    maxIter=20,
    regParam=0.1,
    elasticNetParam=0.0
)

lr_pipeline = Pipeline(stages=indexers + encoders + [assembler, lr])
lr_model = lr_pipeline.fit(train_df)
lr_pred = lr_model.transform(test_df).cache()
lr_pred.select("arr_delay", "prediction", "dep_delay", "carrier", "origin", "dest").show(10, truncate=False)


## Model 2: Random Forest Regression

Parametrii sunt mici ca sa ruleze pe laptop slab.


In [ ]:
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="arr_delay",
    numTrees=25,
    maxDepth=6,
    seed=42
)

rf_pipeline = Pipeline(stages=indexers + encoders + [assembler, rf])
rf_model = rf_pipeline.fit(train_df)
rf_pred = rf_model.transform(test_df).cache()
rf_pred.select("arr_delay", "prediction", "dep_delay", "carrier", "origin", "dest").show(10, truncate=False)


## Metrici: RMSE, MAE, R2


In [ ]:
def regression_metrics(pred_df, model_name):
    evaluators = {
        "RMSE": RegressionEvaluator(labelCol="arr_delay", predictionCol="prediction", metricName="rmse"),
        "MAE": RegressionEvaluator(labelCol="arr_delay", predictionCol="prediction", metricName="mae"),
        "R2": RegressionEvaluator(labelCol="arr_delay", predictionCol="prediction", metricName="r2"),
    }
    rows = []
    for metric_name, evaluator in evaluators.items():
        rows.append((model_name, metric_name, float(evaluator.evaluate(pred_df))))
    return rows

metrics_rows = regression_metrics(lr_pred, "Linear Regression") + regression_metrics(rf_pred, "Random Forest")
metrics_df = spark.createDataFrame(metrics_rows, ["model", "metric", "value"])
metrics_df.show(truncate=False)


Interpretare: RMSE si MAE mai mici inseamna predictii mai bune. R2 mai aproape de 1 inseamna ca modelul explica mai bine variatia intarzierii la sosire. `dep_delay` este de obicei cea mai importanta variabila, deoarece intarzierea la plecare influenteaza direct intarzierea la sosire.


## Comparatie vizuala a metricilor


In [ ]:
import matplotlib.pyplot as plt

pdf_metrics = metrics_df.toPandas()
for metric in ["RMSE", "MAE", "R2"]:
    part = pdf_metrics[pdf_metrics["metric"] == metric]
    plt.figure(figsize=(6, 4))
    plt.bar(part["model"], part["value"])
    plt.title(metric)
    plt.ylabel(metric)
    plt.xticks(rotation=15)
    plt.show()


## Salvare predictii sample

Salvam un sample mic CSV in HDFS pentru demonstratie. Nu salvam Parquet.


In [ ]:
OUTPUT_PRED = "hdfs://master:9000/flights/output/rf_predictions_sample_csv"

rf_pred.select("arr_delay", "prediction", "dep_delay", "air_time", "distance", "carrier", "origin", "dest") \
    .limit(1000) \
    .coalesce(1) \
    .write.mode("overwrite") \
    .option("header", True) \
    .csv(OUTPUT_PRED)

print("Saved predictions sample to:", OUTPUT_PRED)


## Concluzie ML

Am antrenat doua modele de regresie pe Spark MLlib. Linear Regression este simplu si interpretabil. Random Forest poate captura relatii neliniare, dar consuma mai multe resurse. Comparatia finala se face prin RMSE, MAE si R2.
